<a href="https://colab.research.google.com/github/TAUforPython/LLM_AI_agents/blob/main/lesson-1_LLM_base_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medical LLM

- How to Use LLMs
  - Local LLM Inference - CPU & GPU
  - Inference via external API (Mistral example)
- Key takeaways about LLM through examples
  - Hallucinations
  - LLM Strengths
  - LLM Weaknesses

## 0. Environment setup for GPU Colab

Go to Runtime → Change runtime type →  T4 GPU
The T4 has 16GB of VRAM. If you are loading a large model (like a 13B or 30B parameter model) and it crashes, try reducing n_gpu_layers to a number like 20 or 30 instead of -1

In [4]:
!pip install mistralai --quiet

In [5]:
from mistralai.client import Mistral

Here You can get API token: https://admin.mistral.ai/organization/api-keys

In [6]:
import os
import json
import random
import time
from tenacity import retry, stop_after_attempt, wait_exponential


from google.colab import userdata

# Get Hugging Face and PubMed token from environment
MISTRAL_TOKEN = userdata.get("Mistral_API")
if MISTRAL_TOKEN:
  print("✅ MISTRAL token detected")
else:
  print("⚠️  No MISTRAL TOKEN")

✅ MISTRAL token detected


In [7]:
# Set parameters and prompts that we will use through notebook

TEMPERATURE = 0.5
MAX_TOKENS = 400
N_CTX = 2048
SYSTEM_PROMPT = "You're my personal medical assistant. Always start your response by saying 'Hello, patient!'"
USER_QUERY = "Give me 3 ideas where AI agents are useful in healthcare."

PLAIN_PROMPT = f"""\
  System: {SYSTEM_PROMPT}\n
  User: {USER_QUERY}\n
  Assistant:
"""

MESSAGES_PROMPT = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUERY},
]

In [8]:
client = Mistral(api_key=MISTRAL_TOKEN) # Initialize Mistral client

MODEL_NAME = "mistral-small-2503" # Or "open-mistral-7b", "open-mixtral-8x7b", etc.

In [15]:
    prompt = "Generate a short, realistic paragraph (3-5 sentences) about a medical topic."
    chat_response = client.chat.complete(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant providing accurate medical information."},
            {"role": "user", "content": prompt}
        ],
        # Adjust parameters as needed
        max_tokens=500,
        temperature=0.7
    )

In [16]:
chat_response.choices[0].message.content.strip()

'A study published in the *Journal of the American Medical Association* found that regular physical activity, such as brisk walking for 30 minutes a day, can significantly reduce the risk of cardiovascular disease by up to 35%. Exercise helps lower blood pressure, improve cholesterol levels, and maintain a healthy weight, all of which contribute to better heart health. Even moderate activity, like gardening or cycling, can provide measurable benefits. Experts recommend combining aerobic exercise with strength training for optimal results. Small, consistent efforts can lead to long-term health improvements.'

### 1. Inference via external API

In this section, we will use the OpenAI API as an example. You will learn how to work with Yandex Cloud AI Studio in the next lessons.

#### 1.2.1 Setup
First, [generate](https://platform.openai.com/api-keys) an API key and save it in Secrets:
- In the left sidebar, click 🔑 Secrets
- Add a new secret:

```
Name: OPENAI_API_KEY
Value: sk-...
```


Or just paste it below.

In [36]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#### 1.2.2 Call OpenAI with the same prompts and settings


In [30]:
from openai import OpenAI

In [38]:
client = OpenAI()
MODEL = "gpt-5-nano"

def call_openai(messages, model=MODEL):
    resp = client.responses.create(
        model=model,
        input=messages,
    )
    return resp.output_text


print("\n--- OpenAI Chat roles output ---")
print(call_openai(MESSAGES_PROMPT))



--- OpenAI Chat roles output ---
Hello, patient! Here are three ideas where AI agents are useful in healthcare:

- Diagnostic support and imaging analysis: AI helps interpret medical images (X-ray, CT, MRI) and integrate data from tests to generate differential diagnoses, flag potential abnormalities, and prioritize cases for human review. Example: early detection of lung nodules on chest CT, or automated mammography screening triage.

- Personalized medicine and treatment optimization: AI analyzes genomic data, biomarkers, and patient history to predict treatment responses and tailor therapies and dosing, aiming to maximize effectiveness while minimizing adverse effects. Example: selecting targeted therapies in oncology and adjusting dosing based on predicted toxicity.

- Remote monitoring and virtual care: AI-driven chatbots and sensor data monitor patients’ vitals and symptoms between visits, triage concerns, provide coaching, and alert clinicians to deterioration. Example: post-op

## 2. Key takeaways about LLM through examples

### 2.1. Hallucinations

The model may confidently generate false information.

In [42]:
def show(title, text):
  print(f"\n=== {title} ===\n{text}\n")

In [43]:
# ⚠️ DANGEROUS EXAMPLE - DO NOT USE IN PRODUCTION ⚠️
messages = [
    {"role": "system", "content": "You are a medical assistant."},
    {"role": "user", "content": "What is the standard adult dosage for Medicure-500?"}
]

# The AI might hallucinate a specific dosage for a non-existent or real drug
text = call_openai(messages, model="gpt-5-nano")

# 🚨 HALLUCINATED OUTPUT (Example) 🚨
print(text)
# Output: "The standard dosage is 500mg every 4 hours, not exceeding 4000mg per day."

text = call_openai(messages, model="gpt-5-nano")
show("Hallucination risk: MediCure-500 does not exist. The AI made up the name and the dosage confidently.", text)

I’m not sure what “Medicure-500” refers to. I can’t determine a standard adult dosage without knowing the active ingredient and the exact product (strength, form, and route).

Could you provide:
- The active ingredient (or the generic name)
- The strength (e.g., 500 mg) and form (tablet, capsule, solution)
- The route (oral, injectable)
- Any labeling info (imprint code on the pill) or a photo of the bottle

If you have a prescription or bottle label, share that and I can help interpret the dosing. In the meantime, don’t change the dose on your own—follow the label or your prescriber’s instructions, and contact a clinician if you’re unsure or have side effects.

=== Hallucination risk: MediCure-500 does not exist. The AI made up the name and the dosage confidently. ===
I’m not sure what “Medicure-500” refers to, because the dosage depends on the active ingredient inside the medication. Could you tell me the generic (active) name of the drug, or share the exact label/ingredient list? If